# Type 0 (image-level) intervention: stepwise curves

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import torch

from utils import obj_enc_aggregation
from intervention_image_level import (
    build_laion_lexicon, load_intervention_testset, load_gt_categories, load_category_id_by_name,
    load_checkpoint, load_variants, load_boundary_type, apply_intervention,
    balanced_accuracy, positive_variant_accuracy, nb_accuracy,
)
from intervention_image_level_coco import build_trivial_lexicon

ENC = "../encodings/cocologic"
OBJ = "obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25"
TRAIN_JSON = "../data/cocologic/cocologic_train.json"
TEST_JSON = "../data/cocologic/cocologic_test.json"
K = 7
AGGREGATIONS = ["sum", "max"]

VOCABS = {
    "laion": {
        "lexicon": build_laion_lexicon(),
        "base": f"{ENC}/clip/splice_embeddings_0.25.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}.h5",
        "ckpt_tag": "",
    },
    "coco_categories": {
        "lexicon": build_trivial_lexicon("../datasets/coco/annotations/instances_val2017.json"),
        "base": f"{ENC}/clip/splice_embeddings_0.25_coco_categories.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}_coco_categories.h5",
        "ckpt_tag": "_coco_categories",
    },
}
for vocab, v in VOCABS.items():
    v["targets"] = {
        agg: torch.load(f"../data/cocologic/percentile_targets_full_train_{agg}_{vocab}.pt")
        for agg in AGGREGATIONS
    }

category_id_by_name = load_category_id_by_name(TRAIN_JSON)

RULE_NAMES = ["Signal and Ride", "Double Serving", "Herd Alone", "Dog or Car", "Three of a Kind",
              "Car Majority", "Empty Seat", "Single Mode Traffic", "Personal Transport", "Surf Trip"]
PERSON_RULES = ["Herd Alone", "Empty Seat", "Personal Transport", "Surf Trip"]

In [ ]:
test_ids = None
for vocab, v in VOCABS.items():
    emb, obj, img_ids, labels = load_intervention_testset(v["base"], v["obj"], K)
    v["test"] = {"emb": emb, "obj": obj[:, :K]}
    v["raw_min_test"] = min(emb.min().item(), obj.min().item())
    if test_ids is None:
        test_ids, test_labels = img_ids, labels.float()
    assert torch.equal(img_ids, test_ids)

gt_test = load_gt_categories(TEST_JSON, test_ids)
variants = load_variants(TEST_JSON, test_ids)
boundary_type = load_boundary_type(TEST_JSON, test_ids)
print(len(test_ids), "test images")

In [ ]:
def per_rule(logits):
    return {
        "B-Acc": balanced_accuracy(logits, test_labels),
        "PosVar-Acc": positive_variant_accuracy(logits, variants),
        "NB-Acc": nb_accuracy(logits, boundary_type),
    }

def rule_mean(logits):
    return {k: 100 * torch.nanmean(v).item() for k, v in per_rule(logits).items()}

def predict(model, x):
    with torch.no_grad():
        return model(x)

outputs, results = {}, []
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        model = load_checkpoint(f"../checkpoints/cocologic/{agg}_super_multilabel_seed42{v['ckpt_tag']}.pt")
        base = obj_enc_aggregation(agg, v["test"]["obj"], v["test"]["emb"], only_objects=False)
        full = apply_intervention(base, gt_test, v["targets"][agg], category_id_by_name, suppress=True)
        logits = {"none": predict(model, base), "boost+suppress": predict(model, full)}
        outputs[(vocab, agg)] = {"model": model, "base": base, "full": full, "logits": logits}
        for cond, lg in logits.items():
            results.append({"vocab": vocab, "agg": agg, "intervention": cond, **rule_mean(lg)})

## Stepwise intervention

In [ ]:
import random
import numpy as np
from intervention_image_level import stepwise_intervention_curve

RELEVANT = ["traffic light", "bicycle", "bus", "train", "bottle", "cup", "pizza", "cow", "elephant", "sheep",
            "person", "dog", "car", "bowl", "truck", "couch", "chair", "motorcycle", "surfboard"]
ALL_80 = list(VOCABS["coco_categories"]["lexicon"])
NO_LAION_GROUP = {
    "19": ["traffic light", "surfboard"],
    "80": ["traffic light", "surfboard", "fire hydrant", "frisbee", "toaster", "hair drier"],
}
SEEDS = [0, 1, 2, 3, 4]
assert len(ALL_80) == 80 and set(RELEVANT) <= set(ALL_80)

def permutations(categories):
    orders = []
    for seed in SEEDS:
        order = list(categories)
        random.Random(seed).shuffle(order)
        orders.append(order)
    return orders

ORDERS = {"19": permutations(RELEVANT), "80": permutations(ALL_80)}

for curve, orders in ORDERS.items():
    print(f"{curve} categories: steps without effective LAION group")
    for seed, order in zip(SEEDS, orders):
        steps = {order.index(c) + 1: c for c in NO_LAION_GROUP[curve]}
        print(f"  seed {seed}:", ", ".join(f"{n} ({c})" for n, c in sorted(steps.items())))

In [ ]:
stepwise = {}
for curve, orders in ORDERS.items():
    for vocab, v in VOCABS.items():
        for agg in AGGREGATIONS:
            out = outputs[(vocab, agg)]
            stepwise[(curve, vocab, agg)] = stepwise_intervention_curve(
                out["model"], out["base"], gt_test, v["targets"][agg], category_id_by_name,
                test_labels, variants, boundary_type, orders=orders, suppress=True,
            )

table2 = pd.DataFrame(results).set_index(["vocab", "agg", "intervention"])
for (curve, vocab, agg), (ns, stats, curves) in stepwise.items():
    for key, metric in [("bacc", "B-Acc"), ("pv", "PosVar-Acc"), ("nb", "NB-Acc")]:
        last = curves[key][:, -1]
        assert np.all(last == last[0]), (curve, vocab, agg, key, last)
        assert np.allclose(100 * curves[key][:, 0], table2.loc[(vocab, agg, "none"), metric])
        if curve == "80":
            assert np.allclose(100 * last[0], table2.loc[(vocab, agg, "boost+suppress"), metric]), \
                (vocab, agg, key, 100 * last[0], table2.loc[(vocab, agg, "boost+suppress"), metric])
print("checks passed")

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"laion": "#2a78d6", "coco_categories": "#eb6834"}
LABELS = {"laion": "LAION", "coco_categories": "COCO categories"}
METRICS = [("bacc", "B-Acc"), ("pv", "PosVar-Acc"), ("nb", "NB-Acc")]

for curve in ORDERS:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
    for row, agg in enumerate(AGGREGATIONS):
        for col, (key, metric) in enumerate(METRICS):
            ax = axes[row, col]
            for vocab in VOCABS:
                ns, stats, _ = stepwise[(curve, vocab, agg)]
                mean, std = 100 * stats[key][0], 100 * stats[key][1]
                ax.plot(ns, mean, color=COLORS[vocab], linewidth=2, label=LABELS[vocab])
                ax.fill_between(ns, mean - std, mean + std, color=COLORS[vocab], alpha=0.2, linewidth=0)
            ax.set_title(f"{metric} ({agg})")
            ax.grid(alpha=0.3)
            if row == 1:
                ax.set_xlabel("number of intervened categories")
            if col == 0:
                ax.set_ylabel("mean over rules (%)")
    axes[0, 0].legend()
    fig.suptitle(f"Stepwise boost + suppress, {curve} categories, mean ± std over 5 orders")
    fig.tight_layout()
    plt.show()

In [ ]:
TABLE_STEPS = {"19": [0, 5, 10, 15, 19], "80": [0, 20, 40, 60, 80]}
for curve, steps in TABLE_STEPS.items():
    rows = []
    for vocab in VOCABS:
        for agg in AGGREGATIONS:
            _, stats, _ = stepwise[(curve, vocab, agg)]
            for key, metric in METRICS:
                row = {"vocab": vocab, "agg": agg, "metric": metric}
                for n in steps:
                    row[f"step {n}"] = f"{100 * stats[key][0][n]:.2f} ± {100 * stats[key][1][n]:.2f}"
                rows.append(row)
    print(f"{curve} categories: mean ± std over 5 orders (%)")
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## Category contributions



In [ ]:
def mark(cat, vocab):
    tags = []
    if cat in RELEVANT:
        tags.append("R")
    if vocab == "laion" and cat in NO_LAION_GROUP["80"]:
        tags.append("noLAION")
    return f"{cat} [{', '.join(tags)}]" if tags else cat

def top_table(deltas):
    rows = []
    for (vocab, agg), d in deltas.items():
        for key, metric in METRICS:
            ranked = sorted(d[key].items(), key=lambda x: x[1], reverse=True)
            for direction, items in [("rises", ranked[:5]), ("drops", ranked[::-1][:5])]:
                row = {"vocab": vocab, "agg": agg, "metric": metric, "top": direction}
                for i, (cat, delta) in enumerate(items):
                    row[f"#{i + 1}"] = f"{mark(cat, vocab)} {delta:+.2f}"
                rows.append(row)
    return pd.DataFrame(rows)

# a) mean delta when a category is added
added = {}
for curve in ORDERS:
    added[curve] = {}
    for vocab in VOCABS:
        for agg in AGGREGATIONS:
            _, _, curves = stepwise[(curve, vocab, agg)]
            d = {}
            for key, _ in METRICS:
                per_cat = {}
                for order, values in zip(ORDERS[curve], curves[key]):
                    for n, cat in enumerate(order, start=1):
                        per_cat.setdefault(cat, []).append(100 * (values[n] - values[n - 1]))
                d[key] = {cat: np.mean(v) for cat, v in per_cat.items()}
            added[curve][(vocab, agg)] = d

# b) single intervention: only this category corrected, delta vs. no intervention
single = {}
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        out = outputs[(vocab, agg)]
        none = rule_mean(out["logits"]["none"])
        d = {key: {} for key, _ in METRICS}
        for cat in ALL_80:
            targets_cat = {cat: v["targets"][agg][cat]} if cat in v["targets"][agg] else {}
            x = apply_intervention(out["base"], gt_test, targets_cat, category_id_by_name, suppress=True)
            m = rule_mean(predict(out["model"], x))
            for key, metric in METRICS:
                d[key][cat] = m[metric] - none[metric]
        single[(vocab, agg)] = d

pd.set_option("display.width", 250)
print("R = rule-relevant, noLAION = no effective LAION group (LAION rows only); delta in pp")
for curve in ORDERS:
    print(f"\na) mean delta when added, {curve} curve, averaged over 5 orders")
    print(top_table(added[curve]).to_string(index=False))
print("\nb) single intervention (only this category), delta vs. no intervention")
print(top_table(single).to_string(index=False))